# LeetCode 3734 — Lexicographically Smallest Palindromic Permutation Greater Than Target bằng Python

- Link: https://leetcode.com/problems/lexicographically-smallest-palindromic-permutation-greater-than-target/
- Độ khó: **Hard**
- Chủ đề: Chuỗi đối xứng, đếm tần suất, **tham lam từ phải sang trái** (giống `next_permutation`), so sánh từ điển

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def lexPalindromicPermutation(self, s: str, target: str) -> str:
```

## 1. Mục tiêu học

1. Điều kiện tồn tại hoán vị đối xứng: tối đa **một** ký tự có tần suất lẻ.
2. Một palindrome được xác định hoàn toàn bởi **nửa trái** `L` (và ký tự giữa nếu `n` lẻ) — và thứ tự từ điển của palindrome **trùng** thứ tự của `L`.
3. Quy bài toán về: tìm `L` nhỏ nhất (là hoán vị của "nửa đa tập") sao cho `pal(L) > target`.
4. Kỹ thuật "**tiền tố trùng dài nhất, rồi tăng một vị trí, phần sau sắp tăng dần**" — tổng quát hoá `next_permutation` cho đa tập và mục tiêu bất kỳ.
5. Kiểm chứng bằng sinh mọi hoán vị.

## 2. Đề bài & ví dụ

Cho `s`, `target` cùng độ dài `n`. Trả về chuỗi nhỏ nhất theo từ điển vừa là **hoán vị đối xứng** của `s` vừa **lớn hơn hẳn** `target`; không có thì trả `""`.

**Ví dụ 1:** `s = "baba", target = "abba"` → `"baab"`.

**Ví dụ 2:** `s = "baba", target = "bbaa"` → `""`.

**Ví dụ 3:** `s = "abc", target = "abb"` → `""` (không có palindrome).

**Ví dụ 4:** `s = "aac", target = "abb"` → `"aca"`.

**Ràng buộc:** `1 ≤ n ≤ 300`.

In [1]:
from collections import Counter
from itertools import permutations
import random, time

examples = [("baba", "abba", "baab"), ("baba", "bbaa", ""), ("abc", "abb", ""), ("aac", "abb", "aca")]
for s, t, exp in examples:
    print(s, t, "-> kỳ vọng", repr(exp))

baba abba -> kỳ vọng 'baab'
baba bbaa -> kỳ vọng ''
abc abb -> kỳ vọng ''
aac abb -> kỳ vọng 'aca'


## 3. Kiến thức nền

### 3.1. Khi nào có hoán vị đối xứng?

Palindrome ghép cặp ký tự ở vị trí `i` và `n−1−i` ⇒ mọi ký tự xuất hiện số lần chẵn, trừ tối đa một ký tự (ở giữa, khi `n` lẻ).

In [2]:
def half_and_mid(s):
    cnt = Counter(s)
    odd = [c for c in cnt if cnt[c] % 2]
    if len(odd) > 1:
        return None, None
    half = Counter({c: v // 2 for c, v in cnt.items() if v // 2})
    return half, (odd[0] if odd else "")

print(half_and_mid("baba"), half_and_mid("aac"), half_and_mid("abc"))

(Counter({'b': 1, 'a': 1}), '') (Counter({'a': 1}), 'c') (None, None)


### 3.2. Palindrome và nửa trái

`pal(L) = L + mid + rev(L)`. Với hai nửa trái khác nhau `L₁ < L₂` (cùng độ dài), vị trí khác nhau đầu tiên nằm trong nửa trái ⇒ `pal(L₁) < pal(L₂)`. Vậy sắp xếp palindrome ⇔ sắp xếp nửa trái.

In [3]:
build = lambda L, mid: L + mid + L[::-1]
print(build("ab", ""), build("ba", ""), build("a", "c"))

abba baab aca


### 3.3. Chuỗi nhỏ nhất lớn hơn `T` dùng đúng đa tập ký tự

Gọi `m = |T|`. Chuỗi `L > T` nhỏ nhất (cùng đa tập cho trước) có dạng: giữ `T[:pos]`, đặt tại `pos` ký tự **nhỏ nhất còn lại lớn hơn `T[pos]`**, rồi phần còn lại **sắp tăng dần**; chọn `pos` **lớn nhất** có thể (giữ tiền tố trùng dài nhất).

## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Đặt `m = n // 2`, `T = target[:m]`. Vì `pal(L)` và `target` cùng độ dài:

- nếu `L > T` ⇒ `pal(L) > target` (khác nhau trong `m` ký tự đầu);
- nếu `L = T` ⇒ cần so phần còn lại: `pal(T) > target`?
- nếu `L < T` ⇒ `pal(L) < target`.

Vậy: **(a)** nếu `T` dùng được (là hoán vị của nửa đa tập) và `pal(T) > target` ⇒ đáp án `pal(T)`; **(b)** ngược lại tìm `L > T` nhỏ nhất (mục 3.3); **(c)** không có ⇒ `""`.

Ví dụ 1: nửa đa tập `{a, b}`, `T = "ab"`, `pal("ab") = "abba"` không `> "abba"`; `L > "ab"` nhỏ nhất là `"ba"` ⇒ `"baab"`. ✔

## 5. Cách 1 — Brute Force: sinh mọi hoán vị đối xứng

In [4]:
def brute(s, target):
    half, mid = half_and_mid(s)
    if half is None:
        return ""
    letters = "".join(c * v for c, v in sorted(half.items()))
    best = ""
    for p in set(permutations(letters)):
        cand = build("".join(p), mid)
        if cand > target and (best == "" or cand < best):
            best = cand
    return best

for s, t, exp in examples:
    print(repr(brute(s, t)), "kỳ vọng", repr(exp))

'baab' kỳ vọng 'baab'
'' kỳ vọng ''
'' kỳ vọng ''
'aca' kỳ vọng 'aca'


**Nút thắt:** `(n/2)!` hoán vị — chỉ dùng được khi `n ≤ 16`.

## 6. Ý tưởng thuật toán / bước đột phá

1. Tính `half` (26 bộ đếm) và `mid`; nếu > 1 ký tự lẻ ⇒ `""`.
2. Tìm `p` = độ dài tiền tố dài nhất của `T` đặt được bằng `half`.
3. Nếu `p = m` và `pal(T) > target` ⇒ trả về.
4. Với `pos` từ `min(p, m−1)` **giảm** về `0`: với số đếm còn lại sau khi đặt `T[:pos]`, tìm ký tự nhỏ nhất `c > T[pos]` còn dư; nếu có ⇒ `L = T[:pos] + c + (phần còn lại sắp tăng)`, trả `pal(L)`.
5. Không có ⇒ `""`.

Duy trì số đếm **tăng dần** khi lùi `pos` (trả lại `T[pos−1]`) ⇒ `O(26·n)`.

## 7. Các cách tiếp theo

- **Cách 2:** tham lam nhưng tính lại số đếm cho từng `pos` — `O(n² + 26n)`.
- **Cách 3:** tham lam với số đếm cập nhật dần — `O(26 n)` (lời giải nộp).
- **Cách 4:** khi `T` **là** một hoán vị của nửa đa tập, đáp án (nếu `pal(T)` không đủ) chính là **`next_permutation(T)`** — cài đặt kinh điển để đối chiếu.

### 7.1. Cách 2 — Tham lam, tính lại số đếm

In [5]:
A = ord("a")

def greedy_recount(s, target):
    n = len(s); m = n // 2
    half, mid = half_and_mid(s)
    if half is None: return ""
    base = [0] * 26
    for c, v in half.items(): base[ord(c) - A] = v
    T = target[:m]
    for pos in range(m, -1, -1):              # pos = m: thử L = T
        rem = base[:]; ok = True
        for ch in T[:pos]:
            rem[ord(ch) - A] -= 1
            if rem[ord(ch) - A] < 0: ok = False; break
        if not ok: continue
        if pos == m:
            cand = build(T, mid)
            if cand > target: return cand
            continue
        for c in range(ord(T[pos]) - A + 1, 26):
            if rem[c]:
                rem[c] -= 1
                L = T[:pos] + chr(A + c) + "".join(chr(A + x) * rem[x] for x in range(26))
                return build(L, mid)
    return ""

for s, t, exp in examples:
    print(repr(greedy_recount(s, t)), "kỳ vọng", repr(exp))

'baab' kỳ vọng 'baab'
'' kỳ vọng ''
'' kỳ vọng ''
'aca' kỳ vọng 'aca'


### 7.2. Cách 3 — Tham lam `O(26 n)` (lời giải chính)

In [6]:
def greedy_fast(s, target):
    n = len(s); m = n // 2
    half, mid = half_and_mid(s)
    if half is None: return ""
    rem = [0] * 26
    for c, v in half.items(): rem[ord(c) - A] = v
    T = target[:m]
    p = 0
    while p < m and rem[ord(T[p]) - A] > 0:     # đặt tiền tố trùng dài nhất
        rem[ord(T[p]) - A] -= 1; p += 1
    if p == m:
        cand = build(T, mid)
        if cand > target: return cand
        p -= 1 if m > 0 else 0
        if m > 0: rem[ord(T[p]) - A] += 1       # lùi để thử tăng tại vị trí m-1
    for pos in range(min(p, m - 1), -1, -1):
        for c in range(ord(T[pos]) - A + 1, 26):
            if rem[c]:
                rem[c] -= 1
                return build(T[:pos] + chr(A + c) + "".join(chr(A + x) * rem[x] for x in range(26)), mid)
        if pos > 0:
            rem[ord(T[pos - 1]) - A] += 1           # trả lại ký tự trước để thử pos-1
    return ""

for s, t, exp in examples:
    print(repr(greedy_fast(s, t)), "kỳ vọng", repr(exp))

'baab' kỳ vọng 'baab'
'' kỳ vọng ''
'' kỳ vọng ''
'aca' kỳ vọng 'aca'


### 7.3. Cách 4 — `next_permutation` khi `T` là hoán vị hợp lệ

In [7]:
def next_permutation(seq):
    a = list(seq); i = len(a) - 2
    while i >= 0 and a[i] >= a[i + 1]: i -= 1
    if i < 0: return None
    j = len(a) - 1
    while a[j] <= a[i]: j -= 1
    a[i], a[j] = a[j], a[i]; a[i + 1:] = reversed(a[i + 1:])
    return "".join(a)

def via_next_perm(s, target):
    n = len(s); m = n // 2
    half, mid = half_and_mid(s)
    if half is None: return ""
    T = target[:m]
    if Counter(T) != half:
        return greedy_fast(s, target)            # T không phải hoán vị: quay về tham lam
    if build(T, mid) > target: return build(T, mid)
    nxt = next_permutation(T)
    return build(nxt, mid) if nxt is not None else ""

for s, t, exp in examples:
    print(repr(via_next_perm(s, t)), "kỳ vọng", repr(exp))

'baab' kỳ vọng 'baab'
'' kỳ vọng ''
'' kỳ vọng ''
'aca' kỳ vọng 'aca'


## 8. TRACE từng bước

In [8]:
def trace(s, target):
    n = len(s); m = n // 2
    half, mid = half_and_mid(s)
    print(f"s={s}, target={target}, nửa đa tập={dict(sorted(half.items())) if half else None}, giữa={mid!r}")
    if half is None: print("-> > 1 ký tự lẻ: ''"); return
    T = target[:m]; rem = Counter(half); p = 0
    while p < m and rem[T[p]] > 0: rem[T[p]] -= 1; p += 1
    print(f"T = {T!r}, tiền tố đặt được dài p = {p}")
    if p == m:
        print(f"  thử L = T: pal = {build(T, mid)} {'>' if build(T, mid) > target else '<='} target")
    for pos in range(min(p, m - 1), -1, -1):
        r = Counter(half); r.subtract(T[:pos])
        bigger = sorted(c for c in r if r[c] > 0 and c > T[pos])
        print(f"  pos={pos}: giữ {T[:pos]!r}, ký tự còn > {T[pos]!r}: {bigger}")
        if bigger:
            c = bigger[0]; r[c] -= 1
            L = T[:pos] + c + "".join(ch * r[ch] for ch in sorted(r) if r[ch] > 0)
            print(f"  -> L = {L!r}, đáp án {build(L, mid)!r}"); return
    print("  -> không có: ''")

trace("baba", "abba")
trace("aabbccd", "abcdcba")
trace("aabbcc", "cbaabc")

s=baba, target=abba, nửa đa tập={'a': 1, 'b': 1}, giữa=''
T = 'ab', tiền tố đặt được dài p = 2
  thử L = T: pal = abba <= target
  pos=1: giữ 'a', ký tự còn > 'b': []
  pos=0: giữ '', ký tự còn > 'a': ['b']
  -> L = 'ba', đáp án 'baab'
s=aabbccd, target=abcdcba, nửa đa tập={'a': 1, 'b': 1, 'c': 1}, giữa='d'
T = 'abc', tiền tố đặt được dài p = 3
  thử L = T: pal = abcdcba <= target
  pos=2: giữ 'ab', ký tự còn > 'c': []
  pos=1: giữ 'a', ký tự còn > 'b': ['c']
  -> L = 'acb', đáp án 'acbdbca'
s=aabbcc, target=cbaabc, nửa đa tập={'a': 1, 'b': 1, 'c': 1}, giữa=''
T = 'cba', tiền tố đặt được dài p = 3
  thử L = T: pal = cbaabc <= target
  pos=2: giữ 'cb', ký tự còn > 'a': []
  pos=1: giữ 'c', ký tự còn > 'b': []
  pos=0: giữ '', ký tự còn > 'c': []
  -> không có: ''


## 9. Code nộp LeetCode

In [9]:
class Solution:
    def lexPalindromicPermutation(self, s: str, target: str) -> str:
        n = len(s)
        m = n // 2
        cnt = [0] * 26
        for ch in s:
            cnt[ord(ch) - 97] += 1
        odd = [i for i in range(26) if cnt[i] % 2]
        if len(odd) > 1:
            return ""
        mid = chr(97 + odd[0]) if odd else ""
        rem = [v // 2 for v in cnt]

        def build(L):
            return L + mid + L[::-1]

        T = target[:m]
        p = 0
        while p < m and rem[ord(T[p]) - 97] > 0:
            rem[ord(T[p]) - 97] -= 1
            p += 1
        if p == m:
            cand = build(T)
            if cand > target:
                return cand
            if m == 0:
                return ""
            p = m - 1
            rem[ord(T[p]) - 97] += 1
        for pos in range(p, -1, -1):
            for c in range(ord(T[pos]) - 96, 26):
                if rem[c]:
                    rem[c] -= 1
                    tail = "".join(chr(97 + x) * rem[x] for x in range(26))
                    return build(T[:pos] + chr(97 + c) + tail)
            if pos > 0:
                rem[ord(T[pos - 1]) - 97] += 1
        return ""

sol = Solution()
for s, t, exp in examples:
    got = sol.lexPalindromicPermutation(s, t)
    print(s, t, "->", repr(got), "✔" if got == exp else "✘")

baba abba -> 'baab' ✔
baba bbaa -> '' ✔
abc abb -> '' ✔
aac abb -> 'aca' ✔


## 10. Vì sao thuật toán đúng

1. **Tồn tại:** hoán vị đối xứng có ⇔ tối đa một tần suất lẻ (3.1).
2. **Đơn điệu:** `L ↦ pal(L)` bảo toàn thứ tự từ điển (3.2) ⇒ palindrome nhỏ nhất `> target` ứng với `L` nhỏ nhất thoả `pal(L) > target`.
3. **Phân loại `L`:** so với `T = target[:m]`: `L > T` ⇒ thoả; `L < T` ⇒ không thoả; `L = T` ⇒ thoả ⇔ `pal(T) > target`. Vì mọi `L > T` đều lớn hơn `T`, nếu `L = T` thoả thì nó là nhỏ nhất.
4. **`L > T` nhỏ nhất:** `L` và `T` trùng tới `pos` rồi `L[pos] > T[pos]`. `pos` càng lớn thì `L` càng nhỏ (vì tại vị trí `pos'` < `pos` hai chuỗi cùng `T[pos']`, còn chuỗi với `pos'` nhỏ hơn có `L[pos'] > T[pos']`). Với `pos` cố định, chọn `L[pos]` nhỏ nhất `> T[pos]` và đuôi tăng dần là nhỏ nhất. `pos` chỉ hợp lệ khi `T[:pos]` đặt được, tức `pos ≤ p`.
5. Cập nhật `rem` khi lùi `pos` giữ đúng "số đếm sau khi đặt `T[:pos]`".

## 11. Phân tích Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. Sinh hoán vị | `O((n/2)! · n)` | `O((n/2)!)` |
| 2. Tham lam tính lại | `O(n² + 26 n)` | `O(n)` |
| 3. Tham lam cập nhật dần | `O(26 n)` | `O(n)` |
| 4. `next_permutation` | `O(n)` (+ fallback) | `O(n)` |

## 12. Bộ test

In [10]:
METHODS = [brute, greedy_recount, greedy_fast, via_next_perm, lambda s, t: Solution().lexPalindromicPermutation(s, t)]
for s, t, exp in examples:
    assert all(f(s, t) == exp for f in METHODS)
print("Ví dụ đề: OK")

Ví dụ đề: OK


In [11]:
# 12.2 ca biên
for s, t, exp in [("a", "a", ""), ("b", "a", "b"), ("aa", "aa", ""), ("bb", "ba", "bb"),
                  ("bb", "ab", "bb"), ("aba", "aba", ""), ("aab", "aaa", "aba"), ("ccbbaa", "zzzzzz", "")]:
    assert all(f(s, t) == exp for f in METHODS), (s, t, [f(s, t) for f in METHODS])
print("Ca biên: OK")

Ca biên: OK


In [12]:
# 12.3 ngẫu nhiên: target ngẫu nhiên hoặc là một hoán vị/palindrome lân cận
rng = random.Random(3734)
for _ in range(3000):
    n = rng.randint(1, 10)
    half = [rng.choice("abc") for _ in range(n // 2)]
    mid = [rng.choice("abc")] if n % 2 else []
    s = list(half * 2 + mid); rng.shuffle(s); s = "".join(s)
    if rng.random() < 0.2:
        s = "".join(rng.choice("abc") for _ in range(n))        # có thể không có palindrome
    r = rng.random()
    if r < 0.4:
        t = "".join(rng.choice("abcd") for _ in range(n))
    else:
        h = half[:]; rng.shuffle(h); t = "".join(h) + "".join(mid) + "".join(h[::-1])
        if r < 0.7 and n:
            i = rng.randrange(n); t = t[:i] + rng.choice("abcd") + t[i + 1:]
    b = brute(s, t)
    assert all(f(s, t) == b for f in METHODS[1:]), (s, t, b, [f(s, t) for f in METHODS])
print("3000 test ngẫu nhiên: OK")

3000 test ngẫu nhiên: OK


In [13]:
# 12.4 test lớn: n = 300
rng = random.Random(1)
h = [rng.choice("abcdefghijklmnopqrstuvwxyz") for _ in range(150)]
s = "".join(h * 2); t = "".join(sorted(h)) + "".join(sorted(h))[::-1]
t0 = time.time(); x = Solution().lexPalindromicPermutation(s, t); t1 = time.time()
print("độ dài:", len(x), "| là palindrome:", x == x[::-1], "| > target:", x > t, "| thời gian:", round(t1 - t0, 4), "s")
assert x == via_next_perm(s, t) == greedy_recount(s, t)

độ dài: 300 | là palindrome: True | > target: True | thời gian: 0.0001 s


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Bỏ qua trường hợp `L = T` (chỉ tìm `L > T`)

In [14]:
def wrong_skip_equal(s, target):
    n = len(s); m = n // 2
    half, mid = half_and_mid(s)
    if half is None: return ""
    T = target[:m]
    best = ""
    letters = "".join(c * v for c, v in sorted(half.items()))
    for p in set(permutations(letters)):
        L = "".join(p)
        if L > T and (best == "" or build(L, mid) < best): best = build(L, mid)   # SAI: thiếu L == T
    return best

print("sai:", repr(wrong_skip_equal("aac", "abb")), "| đúng:", repr(brute("aac", "abb")))

sai: '' | đúng: 'aca'


### Lỗi 2 — Dùng `≥` thay vì `>` khi so với `target`

In [15]:
def wrong_ge(s, target):
    half, mid = half_and_mid(s)
    if half is None: return ""
    letters = "".join(c * v for c, v in sorted(half.items()))
    c = sorted(build("".join(p), mid) for p in set(permutations(letters)) if build("".join(p), mid) >= target)  # SAI
    return c[0] if c else ""

print("sai:", repr(wrong_ge("baba", "abba")), "| đúng:", repr(brute("baba", "abba")))

sai: 'abba' | đúng: 'baab'


### Lỗi 3 — Chỉ thử tăng tại vị trí cuối của tiền tố trùng (không lùi)

In [16]:
def wrong_no_backtrack(s, target):
    n = len(s); m = n // 2
    half, mid = half_and_mid(s)
    if half is None or m == 0: return "" if half is None else (s if s > target else "")
    rem = Counter(half); T = target[:m]; p = 0
    while p < m and rem[T[p]] > 0: rem[T[p]] -= 1; p += 1
    if p == m:
        if build(T, mid) > target: return build(T, mid)
        p = m - 1; rem[T[p]] += 1
    bigger = sorted(c for c in rem if rem[c] > 0 and c > T[p])              # SAI: không lùi pos
    if not bigger: return ""
    c = bigger[0]; rem[c] -= 1
    return build(T[:p] + c + "".join(ch * rem[ch] for ch in sorted(rem) if rem[ch] > 0), mid)

print("sai:", repr(wrong_no_backtrack("aabbcc", "acbbca")), "| đúng:", repr(brute("aabbcc", "acbbca")))
assert wrong_no_backtrack("aabbcc", "acbbca") != brute("aabbcc", "acbbca")

sai: '' | đúng: 'baccab'


### Lỗi 4 — Quên kiểm tra số ký tự lẻ

In [17]:
def wrong_no_odd_check(s, target):
    cnt = Counter(s); half = Counter({c: v // 2 for c, v in cnt.items()})
    mid = "".join(c for c in cnt if cnt[c] % 2)[:1]                            # SAI: bỏ ký tự lẻ thừa
    letters = "".join(c * v for c, v in sorted(half.items() ) if v)
    c = sorted(build("".join(p), mid) for p in set(permutations(letters)) if build("".join(p), mid) > target)
    return c[0] if c else ""

print("sai:", repr(wrong_no_odd_check("aabbcd", "aaaaaa")), "| đúng:", repr(brute("aabbcd", "aaaaaa")))
assert wrong_no_odd_check("aabbcd", "aaaaaa") != brute("aabbcd", "aaaaaa")

sai: 'abcba' | đúng: ''


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Palindrome nhỏ nhất thoả …" ⇒ làm việc trên **nửa trái**.
- "Nhỏ nhất lớn hơn `T` với đa tập ký tự cố định" ⇒ tiền tố trùng dài nhất, tăng tại vị trí xa nhất có thể, đuôi tăng dần (tổng quát `next_permutation`).
- Đừng quên trường hợp **bằng** ở nửa trái — khi đó quyết định nằm ở nửa phải.
- `n ≤ 300` nhưng `(n/2)!` hoán vị ⇒ chắc chắn cần tham lam.

## 15. Biến thể của bài

1. **Lớn nhất nhỏ hơn `target`**: đối xứng — tiền tố trùng dài nhất, giảm tại vị trí xa nhất, đuôi giảm dần.
2. **Hoán vị bất kỳ (không cần palindrome)** nhỏ nhất `> target`: áp dụng 3.3 trực tiếp lên cả chuỗi.
3. **Đếm số hoán vị đối xứng `> target`**: đếm hoán vị đa tập với tiền tố cố định (công thức đa thức).
4. **Palindrome thứ `k`**: unrank hoán vị đa tập của nửa trái.

In [18]:
# Biến thể 3: đếm số hoán vị đối xứng lớn hơn target
from math import factorial

def multiset_perms(cnt):
    tot = sum(cnt); r = factorial(tot)
    for v in cnt: r //= factorial(v)
    return r

def count_greater(s, target):
    n = len(s); m = n // 2
    half, mid = half_and_mid(s)
    if half is None: return 0
    rem = [0] * 26
    for c, v in half.items(): rem[ord(c) - A] = v
    T = target[:m]; total = 0
    for pos in range(m):
        for c in range(ord(T[pos]) - A + 1, 26):
            if rem[c]:
                rem[c] -= 1; total += multiset_perms(rem); rem[c] += 1
        x = ord(T[pos]) - A
        if rem[x] == 0: return total
        rem[x] -= 1
    return total + (1 if build(T, mid) > target else 0)

rng = random.Random(15)
for _ in range(500):
    n = rng.randint(1, 9); h = [rng.choice("abc") for _ in range(n // 2)]
    s = "".join(h * 2) + (rng.choice("abc") if n % 2 else ""); t = "".join(rng.choice("abc") for _ in range(n))
    hm, md_ = half_and_mid(s)
    letters = "".join(c * v for c, v in sorted(hm.items()))
    exp = sum(1 for p in set(permutations(letters)) if build("".join(p), md_) > t)
    assert count_greater(s, t) == exp
print("Biến thể 3 OK")

Biến thể 3 OK


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Khi nào dùng |
|---|---|---|---|
| 1. Sinh hoán vị | Thử hết | `O((n/2)!)` | Kiểm chứng |
| 2. Tham lam tính lại | Mỗi `pos` đếm lại | `O(n²)` | Dễ viết đúng |
| 3. Tham lam cập nhật dần | Lùi `pos`, trả ký tự | `O(26 n)` | **Lời giải chuẩn** |
| 4. `next_permutation` | Khi `T` hợp lệ | `O(n)` | Hiểu bản chất |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output).** `s = "aabbc"`, `target = "abcba"`.

**Bài 2 (tính tay).** `s = "aabbcc"`, `target = "acbbca"`.

**Bài 3 (điền TODO).** Viết `largest_smaller(s, target)` — palindrome **lớn nhất nhỏ hơn hẳn** `target`.

**Bài 4 (tự viết lại).** Viết `next_greater_perm(s, target)` cho hoán vị **bất kỳ** (không cần palindrome).

**Bài 5 (mini challenge).** Trả về palindrome thứ `k` (1-indexed) trong thứ tự từ điển của các hoán vị đối xứng của `s`.

In [19]:
# Bài 3 — TODO (không gọi hàm này khi chạy notebook)
def largest_smaller(s, target):
    # TODO
    pass

In [20]:
# Bài 5 — TODO (không gọi hàm này khi chạy notebook)
def kth_palindrome(s, k):
    # TODO: unrank bằng multiset_perms
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) Nửa `{a, b}`, giữa `c`, `T = "ab"`. (2) `pal("ab") = "abcba"` bằng target ⇒ không đủ. (3) `L = "ba"` ⇒ `"bacab"`.

**Bài 2.** (1) `T = "acb"`, đặt được. (2) `pal(T) = "acbbca"` bằng target. (3) Lùi: `pos = 1`, ký tự > `c` không có; `pos = 0`... thực ra `pos = 2`: còn `{}`; `pos = 1`: còn `{b, c}` > `c`? không; `pos = 0`: chọn `b` ⇒ `L = "bac"` ⇒ `"baccab"`.

**Bài 3.** (1) Đối xứng chiều so sánh. (2) Tiền tố trùng dài nhất, giảm tại `pos` xa nhất. (3) Đuôi sắp giảm dần.

**Bài 4.** (1) Không cần nửa. (2) Áp dụng thuật toán trên cả chuỗi với `Counter(s)`. (3) Kiểm chứng bằng `permutations`.

**Bài 5.** (1) Duyệt vị trí, thử ký tự tăng dần. (2) Số chuỗi bắt đầu bằng tiền tố = `multiset_perms(rem)`. (3) Trừ dần `k`.

## 19. Đáp án tham khảo cho bài tập

In [21]:
# Bài 1, 2
assert Solution().lexPalindromicPermutation("aabbc", "abcba") == "bacab" == brute("aabbc", "abcba")
assert Solution().lexPalindromicPermutation("aabbcc", "acbbca") == "baccab" == brute("aabbcc", "acbbca")
print("Bài 1, 2 OK")

Bài 1, 2 OK


In [22]:
# Bài 3, 4
def largest_smaller_sol(s, target):
    half, mid = half_and_mid(s)
    if half is None: return ""
    letters = "".join(c * v for c, v in sorted(half.items()))
    n = len(s); m = n // 2; T = target[:m]
    rem = Counter(half); p = 0
    while p < m and rem[T[p]] > 0: rem[T[p]] -= 1; p += 1
    if p == m:
        if build(T, mid) < target: return build(T, mid)
        if m == 0: return ""
        p = m - 1; rem[T[p]] += 1
    for pos in range(min(p, m - 1), -1, -1):
        smaller = sorted((c for c in rem if rem[c] > 0 and c < T[pos]), reverse=True)
        if smaller:
            c = smaller[0]; rem[c] -= 1
            return build(T[:pos] + c + "".join(ch * rem[ch] for ch in sorted(rem, reverse=True) if rem[ch] > 0), mid)
        if pos > 0: rem[T[pos - 1]] += 1
    return ""

def next_greater_perm_sol(s, target):
    rem = Counter(s); n = len(s); p = 0
    while p < n and rem[target[p]] > 0: rem[target[p]] -= 1; p += 1
    if p == n: p = n - 1; rem[target[p]] += 1                 # bằng target: phải lớn hơn hẳn
    for pos in range(p, -1, -1):
        bigger = sorted(c for c in rem if rem[c] > 0 and c > target[pos])
        if bigger:
            c = bigger[0]; rem[c] -= 1
            return target[:pos] + c + "".join(ch * rem[ch] for ch in sorted(rem) if rem[ch] > 0)
        if pos > 0: rem[target[pos - 1]] += 1
    return ""

rng = random.Random(4)
for _ in range(500):
    n = rng.randint(1, 8); h = [rng.choice("abc") for _ in range(n // 2)]
    s = "".join(h * 2) + (rng.choice("abc") if n % 2 else ""); t = "".join(rng.choice("abc") for _ in range(n))
    hm, md_ = half_and_mid(s); letters = "".join(c * v for c, v in sorted(hm.items()))
    pals = sorted(build("".join(p), md_) for p in set(permutations(letters)))
    smaller = [x for x in pals if x < t]
    assert largest_smaller_sol(s, t) == (smaller[-1] if smaller else "")
    perms = sorted(set("".join(p) for p in permutations(s)))
    gt = [x for x in perms if x > t]
    assert next_greater_perm_sol(s, t) == (gt[0] if gt else "")
print("Bài 3, 4 OK")

Bài 3, 4 OK


In [23]:
# Bài 5
def kth_palindrome_sol(s, k):
    half, mid = half_and_mid(s)
    if half is None: return ""
    rem = [0] * 26
    for c, v in half.items(): rem[ord(c) - A] = v
    if k > multiset_perms(rem): return ""
    L = []
    for _ in range(sum(rem)):
        for c in range(26):
            if rem[c]:
                rem[c] -= 1; cnt = multiset_perms(rem)
                if k <= cnt: L.append(chr(A + c)); break
                k -= cnt; rem[c] += 1
    return build("".join(L), mid)

hm, md_ = half_and_mid("aabbccd")
pals = sorted(build("".join(p), md_) for p in set(permutations("".join(c * v for c, v in sorted(hm.items())))))
assert [kth_palindrome_sol("aabbccd", k) for k in range(1, len(pals) + 1)] == pals
print("Bài 5 OK:", pals[:3], "...", len(pals), "palindrome")

Bài 5 OK: ['abcdcba', 'acbdbca', 'bacdcab'] ... 6 palindrome


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
# >1 ký tự lẻ -> ""; rem = cnt // 2; T = target[:m]
p = tiền tố T dài nhất đặt được (trừ rem dần)
if p == m and pal(T) > target: return pal(T)
if p == m: p = m-1; trả lại T[p]
for pos = p .. 0:
    c = ký tự nhỏ nhất > T[pos] còn trong rem -> return pal(T[:pos] + c + đuôi tăng dần)
    trả lại T[pos-1]
return ""
```

**Checklist:**

- [ ] Kiểm tra số ký tự lẻ?
- [ ] Xét trường hợp `L = T` (so cả palindrome với target)?
- [ ] So sánh **nghiêm ngặt** `>`?
- [ ] Lùi `pos` và trả lại ký tự đúng?
- [ ] `n = 1` (nửa rỗng) xử lý đúng?